In [8]:
import pandas as pd

# 1. Load the dataset
# Adjust the path relative to your notebook location
file_path = '../dataset/data_andre.feather'
df = pd.read_feather(file_path)

# Ensure the date column is in datetime format
df['date'] = pd.to_datetime(df['date'])

# 2. Extract unique products/stores and the full date range
unique_items = df[['item_id', 'store_id']].drop_duplicates()

# Get the global min and max date to build a continuous daily calendar
min_date = df['date'].min()
max_date = df['date'].max()
full_dates = pd.date_range(start=min_date, end=max_date, freq='D')
dates_df = pd.DataFrame({'date': full_dates})

# 3. Create the complete backbone (cross join)
# This creates a row for every single product-store combination for every single day
full_grid = unique_items.merge(dates_df, how='cross')

# 4. Merge the original data back onto the backbone
df_complete = full_grid.merge(df, on=['item_id', 'store_id', 'date'], how='left')

# 5. Impute the explicit missing rows!
# Newly created rows will have NaNs for targets/features. 
# Depending on your use case, you might want to fill gaps in your target with 0:
df_complete['value'] = df_complete['value'].fillna(0)

# (Optional) Verify the results
print(f"Original rows: {len(df)}")
print(f"Rows after filling days: {len(df_complete)}")

# (Optional) Save back to feather
# df_complete.reset_index(drop=True).to_feather('../dataset/data_andre_full_days.feather')

Original rows: 1082371
Rows after filling days: 1085947


In [9]:
import pandas as pd

# 1. Load the dataset (adjust path as needed)
df = pd.read_feather('../dataset/data_andre.feather')
df['date'] = pd.to_datetime(df['date'])

# 2. Determine the expected date range (global min and max)
min_date = df['date'].min()
max_date = df['date'].max()
expected_dates = pd.date_range(start=min_date, end=max_date, freq='D')
expected_count = len(expected_dates)

print(f"Global date range: {min_date.date()} to {max_date.date()} ({expected_count} days)")

# 3. IDENTIFY incomplete products
# Count how many unique days each product actually has in the dataset
date_counts = df.groupby(['item_id', 'store_id'])['date'].nunique().reset_index()
date_counts.rename(columns={'date': 'actual_days'}, inplace=True)

# Filter down to the ones that have fewer days than the expected count
incomplete_products = date_counts[date_counts['actual_days'] < expected_count]

print(f"\nTotal product-store combinations: {len(date_counts)}")
print(f"Total incomplete combinations: {len(incomplete_products)}")

if len(incomplete_products) > 0:
    print("\nSample of incomplete products:")
    display(incomplete_products.head(10))

# 4. FULFILL missing dates
# Create a complete backbone for all unique items with the full date range
unique_items = df[['item_id', 'store_id']].drop_duplicates()
dates_df = pd.DataFrame({'date': expected_dates})

# Cross join: every unique item X every expected date
full_grid = unique_items.merge(dates_df, how='cross')

# Merge actual data onto the complete grid (missing dates will become rows with NaNs)
df_fulfilled = full_grid.merge(df, on=['item_id', 'store_id', 'date'], how='left')

# 5. Fill the newly spawned NaNs in the target variable
# Assuming your target variable is named 'value'
df_fulfilled['value'] = df_fulfilled['value'].fillna(0)

print(f"\nOriginal row count: {len(df)}")
print(f"Fulfilled row count: {len(df_fulfilled)}")

# (Optional) Verify that the fix worked:
missing_check = df_fulfilled.groupby(['item_id', 'store_id'])['date'].nunique().min()
print(f"Minimum days any product has after fulfillment: {missing_check} (Should be {expected_count})")


Global date range: 2021-01-23 to 2023-02-22 (761 days)

Total product-store combinations: 1427
Total incomplete combinations: 467

Sample of incomplete products:


,item_id,store_id,actual_days
6,176,6269,760
14,260,6269,759
28,471,6269,739
31,493,6269,759
34,514,6269,751
37,612,6269,760
41,676,6269,760
48,779,6269,734
50,808,6269,760
65,1241,6269,759



Original row count: 1082371
Fulfilled row count: 1085947
Minimum days any product has after fulfillment: 761 (Should be 761)


In [10]:
# 6. Save the fulfilled dataset
df_fulfilled.reset_index(drop=True).to_feather('../dataset/data_andre_fulfilled.feather')